# Projet — Classification d''images Intel (Hugging Face)

**Noyau** : `deep5` (Python 3.12, TensorFlow 2.21)

**Dataset** : [`sfarrukhm/intel-image-classification`](https://huggingface.co/datasets/sfarrukhm/intel-image-classification)

**Classes (6)** : `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`

**Travail realise** :
- [ ] Q1 — Chargement et exploration des donnees
- [ ] Q2 — Etude de l''equilibre des classes
- [ ] Q3 — Construction d''un CNN from-scratch
- [ ] Q4 — Entrainement et evaluation
- [ ] Q5 — GridSearchCV (hyperparametres)
- [ ] Q6 — Augmentation d''images
- [ ] Q7 — Sauvegarde du meilleur modele
- [ ] Q8 — Test sur photos personnelles
- [ ] Q9 — Transfer learning (ResNet)
- [ ] Bonus 1 — CAM
- [ ] Bonus 2 — Grad-CAM

## Cellule 0 — Infrastructure commune (a executer en premier)

Cette cellule prepare l''environnement : imports, graines de reproductibilite, et verification des packages.

A executer **une seule fois** en haut du notebook.

In [2]:
# --- Reproductibilite ----------------------------------------------
import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Imports --------------------------------------------------------
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from PIL import Image
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier

# --- Constantes -----------------------------------------------------
TAILLE_IMAGE = (150, 150)
NOMBRE_CLASSES = 6
TAILLE_LOT = 32

labels = []
dataset = None
images_train = labels_train = None
images_test  = labels_test  = None
donnees_entrainement = donnees_test = None
modele_cnn = None

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU dispo   : {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Environnement pret (SEED = {SEED}).")

TensorFlow  : 2.21.0
GPU dispo   : False
Environnement pret (SEED = 42).


In [3]:
import os
cache = os.path.expanduser("~/.cache/huggingface/datasets")
print("Cache :", cache)
os.system(f"du -sh {cache} 2>nul" if os.name != "nt" else f"powershell -Command \"Get-ChildItem '{cache}' | Measure-Object -Property Length -Sum\"")

Cache : C:\Users\KOURO/.cache/huggingface/datasets


1

In [4]:
from pathlib import Path
import os

# Lecture du token comme le fait la cellule de Q1
HF_TOKEN = None
env_path = Path(".env")
if env_path.exists():
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == "HF_TOKEN":
            HF_TOKEN = value.strip().strip('"').strip("'")
            break

print("Token trouvé :", "OUI" if HF_TOKEN else "NON")
print("Longueur :", len(HF_TOKEN) if HF_TOKEN else 0)
print("Préfixe     :", HF_TOKEN[:6] + "..." if HF_TOKEN else "—")
print("CWD actuel :", os.getcwd())

Token trouvé : OUI
Longueur : 37
Préfixe     : hf_nxV...
CWD actuel : c:\Users\KOURO\Desktop\Mini projet


---

# QUESTION 1 — Chargement et exploration

Coller ici les cellules de votre Question 1 issue de `tr.ipynb`.

Conseil : commencer par la cellule de chargement (`load_dataset`), puis la structure, les classes, la visualisation, les tailles, la distribution et les statistiques pixel.